# Investigación aplicada en Deep Learning
## XAI para modelos médicos críticos: ¿en qué se fija una CNN al detectar tuberculosis en rayos X?

**Objetivo.** Entrenar dos CNN con *transfer learning* (DenseNet121 y ResNet50) para clasificar radiografías de tórax (normal vs. tuberculosis), evaluarlas **dentro** (Shenzhen) y **fuera** de distribución (Montgomery) y comprobar, con **Grad-CAM** y **LIME**, de forma visual y *cuantitativa*, si sus decisiones se apoyan en los pulmones o en artefactos (bordes, marcas, texto).

**Dataset.** Kaggle `kmader/pulmonary-chest-xray-abnormalities` (el indicado en la guía): conjuntos *Shenzhen* (~662 imágenes, sin máscaras) y *Montgomery* (138 imágenes, con máscaras manuales de pulmón).

**Diseño experimental**
| Etapa | Qué se hace |
|---|---|
| Entrenamiento | Shenzhen (70 % train / 10 % val / 20 % test, estratificado) |
| Validación externa | Montgomery completo (otro hospital, otro escáner) |
| Modelos | DenseNet121 y ResNet50 preentrenados en ImageNet, 3 semillas cada uno |
| XAI | Grad-CAM (todas las imágenes) y LIME (subconjunto), + métricas cuantitativas con las máscaras de Montgomery |

**Cómo ejecutarlo.** Colab / Kaggle con GPU → *Run all*. El notebook descarga el dataset (con `kagglehub`) o lo busca en `/kaggle/input`. Todo lo que necesita el artículo (tablas `.tex`, macros, figuras) se guarda en `results/`.

**Contenido:** 0 Configuración · 1 Datos · 2 Preprocesamiento y particiones · 3 Modelos · 4 Entrenamiento · 5 Evaluación · 6 Explicabilidad (XAI) · 7 Exportación para el artículo.

## 0. Configuración y reproducibilidad

In [ ]:
# Dependencias instaladas en .venv; usar el kernel 'DeepLinv (CUDA)'.


In [ ]:
import os, sys, json, random, time, warnings, zipfile, platform
from pathlib import Path
warnings.filterwarnings("default")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from tqdm.auto import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import models, transforms as T

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay
from scipy import stats

# --- Hiperparámetros (todo lo configurable está aquí) ---
QUICK = os.environ.get("QUICK_RUN", "0") == "1"       # modo de prueba rápida (solo para depurar)
CFG = dict(
    IMG_SIZE=128 if QUICK else 224, BATCH=4, ACCUM_STEPS=1 if QUICK else 8,
    EPOCHS=3 if QUICK else 20, PATIENCE=5,             # early stopping sobre el AUC de validación
    LR=1e-4, WD=1e-4, DROPOUT=0.3,
    SEEDS=[0] if QUICK else [0, 1, 2],                 # 3 semillas -> media ± desviación
    MODELS=["densenet121", "resnet50"],
    N_XAI=8 if QUICK else 40,                          # imágenes de Montgomery para LIME + fidelidad
    LIME_SAMPLES=60 if QUICK else 800,
    PRETRAINED=os.environ.get("NO_PRETRAINED", "0") != "1",
    SPLIT_SEED=42,
)
OUT = Path(os.environ.get("RESULTS_DIR", "results")); OUT.mkdir(exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_WORKERS = 0 if os.name == "nt" else (2 if device.type == "cuda" else 0)
IMAGENET_MEAN, IMAGENET_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

print(f"Dispositivo: {device} | torch {torch.__version__} | torchvision {torchvision.__version__}")
if device.type == "cpu" and not QUICK:
    print("AVISO: no hay GPU; el entrenamiento será lento. Activa GPU en Entorno de ejecución.")
print(json.dumps(CFG, indent=1))

## 1. Datos: descubrimiento, etiquetas y análisis exploratorio
Las etiquetas están en el nombre de archivo: `..._0.png` = normal, `..._1.png` = tuberculosis (TB). En ambos conjuntos **cada imagen es un paciente distinto**, por lo que dividir por archivo equivale a dividir por paciente (no hay fuga de datos entre particiones).

In [ ]:
def find_data_root():
    cands = [os.environ.get("CXR_DATA_ROOT"),
             "/kaggle/input/pulmonary-chest-xray-abnormalities",
             "./pulmonary-chest-xray-abnormalities", "./data"]
    for c in cands:
        if c and Path(c).exists() and any(Path(c).rglob("CXR_png/*.png")):
            return Path(c)
    import kagglehub                                   # descarga pública, sin credenciales
    return Path(kagglehub.dataset_download("kmader/pulmonary-chest-xray-abnormalities"))

ROOT = find_data_root()
print("Dataset en:", ROOT)

rows, masks_l, masks_r = [], {}, {}
for p in ROOT.rglob("*.png"):
    parts = set(p.parts)
    if "leftMask" in parts:  masks_l[p.name] = str(p); continue
    if "rightMask" in parts: masks_r[p.name] = str(p); continue
    if "CXR_png" not in parts: continue
    src = "montgomery" if p.name.startswith("MCUCXR") else ("shenzhen" if p.name.startswith("CHNCXR") else None)
    if src is None: continue
    rows.append(dict(path=str(p), name=p.name, source=src, label=int(p.stem.split("_")[-1])))

df = pd.DataFrame(rows)
assert not df.empty, "No se encontraron radiografías."
assert not df.name.duplicated().any(), "Hay nombres de radiografía duplicados."
df = df.sort_values("name").reset_index(drop=True)
df["has_mask"] = df.name.isin(masks_l) & df.name.isin(masks_r)
assert set(df.source) == {"shenzhen", "montgomery"}, "No se encontraron ambos conjuntos; revisa la ruta del dataset."
assert (df.source.value_counts().to_dict() == {"shenzhen": 662, "montgomery": 138}), "Conteo de imágenes distinto de la fuente NLM."
assert df.loc[df.source == "montgomery", "has_mask"].all(), "Faltan máscaras de Montgomery."
assert set(df.label) == {0, 1}, "Etiquetas inesperadas."

summary = df.groupby(["source", "label"]).size().unstack(fill_value=0).rename(columns={0: "Normal", 1: "TB"})
summary["Total"] = summary.sum(axis=1); summary["con máscara"] = df.groupby("source").has_mask.sum()
display(summary)

In [ ]:
# Caché en memoria (las radiografías originales son enormes: se redimensionan UNA vez)
S = CFG["IMG_SIZE"]
IMG_CACHE = {p: Image.open(p).convert("L").resize((S, S), Image.BILINEAR) for p in tqdm(df.path, desc="Imágenes")}
MASK_CACHE = {}                                        # máscara de pulmón (izq. U der.), solo Montgomery
for n in tqdm(df[df.has_mask].name, desc="Máscaras"):
    l = np.array(Image.open(masks_l[n]).convert("L").resize((S, S), Image.NEAREST)) > 0
    r = np.array(Image.open(masks_r[n]).convert("L").resize((S, S), Image.NEAREST)) > 0
    MASK_CACHE[n] = l | r

# Muestras por fuente y clase (con contorno de pulmón en Montgomery)
fig, axs = plt.subplots(2, 4, figsize=(11, 5.6))
for r, (src, lab) in enumerate([("shenzhen", 0), ("shenzhen", 1)]):
    for c in range(2):
        sub = df[(df.source == src) & (df.label == lab)].iloc[c * 7]
        axs[r, c].imshow(IMG_CACHE[sub.path], cmap="gray"); axs[r, c].set_title(f"Shenzhen · {'TB' if lab else 'Normal'}", fontsize=9)
for r, lab in enumerate([0, 1]):
    for c in range(2):
        sub = df[(df.source == "montgomery") & (df.label == lab)].iloc[c * 5]
        axs[r, 2 + c].imshow(IMG_CACHE[sub.path], cmap="gray")
        if sub.name in MASK_CACHE: axs[r, 2 + c].contour(MASK_CACHE[sub.name], levels=[0.5], colors="yellow", linewidths=0.8)
        axs[r, 2 + c].set_title(f"Montgomery · {'TB' if lab else 'Normal'}", fontsize=9)
for a in axs.ravel(): a.axis("off")
plt.tight_layout(); plt.savefig(OUT / "fig_samples.png", dpi=200); plt.show()

## 2. Preprocesamiento y particiones
* Escala de grises → 3 canales (las redes preentrenadas esperan RGB), redimensionado a 224×224 y normalización con media/desviación de ImageNet.
* *Data augmentation* **suave y clínicamente plausible**: rotación ≤ 8°, traslación ≤ 4 %, escala ±5 %, brillo/contraste ±15 %. **No** se usa *flip* horizontal: invertiría la posición del corazón y alteraría la anatomía.
* Particiones **estratificadas** y fijas (`SPLIT_SEED`) para que las 3 semillas varíen solo la inicialización y el *augmentation*. Validación externa: Montgomery completo, nunca visto durante el entrenamiento ni la selección de umbral.

In [ ]:
sz = df[df.source == "shenzhen"].reset_index(drop=True)
mg = df[df.source == "montgomery"].reset_index(drop=True)

sz_trval, sz_test = train_test_split(sz, test_size=0.20, stratify=sz.label, random_state=CFG["SPLIT_SEED"])
sz_train, sz_val  = train_test_split(sz_trval, test_size=0.125, stratify=sz_trval.label, random_state=CFG["SPLIT_SEED"])
sz_train, sz_val, sz_test = [d.reset_index(drop=True) for d in (sz_train, sz_val, sz_test)]

split_tbl = pd.DataFrame({k: {"n": len(d), "TB (%)": round(100 * d.label.mean(), 1)}
                          for k, d in {"Shenzhen train": sz_train, "Shenzhen val": sz_val,
                                       "Shenzhen test": sz_test, "Montgomery (externo)": mg}.items()}).T
display(split_tbl)
SPLITS = pd.concat([d.assign(split=name) for name, d in [("train", sz_train), ("val", sz_val), ("test", sz_test), ("external", mg)]], ignore_index=True)
assert SPLITS.name.is_unique, "Hay fuga de imágenes entre particiones."
SPLITS[["name", "source", "label", "split"]].to_csv(OUT / "splits.csv", index=False)

train_tf = T.Compose([T.RandomAffine(degrees=8, translate=(0.04, 0.04), scale=(0.95, 1.05)),
                      T.ColorJitter(brightness=0.15, contrast=0.15),
                      T.Grayscale(3), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])
eval_tf  = T.Compose([T.Grayscale(3), T.ToTensor(), T.Normalize(IMAGENET_MEAN, IMAGENET_STD)])

class CXRDataset(Dataset):
    def __init__(self, frame, train=False):
        self.paths, self.y = frame.path.tolist(), frame.label.values
        self.tf = train_tf if train else eval_tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i): return self.tf(IMG_CACHE[self.paths[i]]), int(self.y[i])

def make_loader(frame, train=False, seed=0, bs=None):
    g = torch.Generator().manual_seed(seed)
    return DataLoader(CXRDataset(frame, train), batch_size=bs or CFG["BATCH"], shuffle=train, generator=g,
                      num_workers=NUM_WORKERS, pin_memory=device.type == "cuda", drop_last=False)

def get_tensor(frame):
    return torch.stack([eval_tf(IMG_CACHE[p]) for p in frame.path])

## 3. Modelos
Dos arquitecturas con *transfer learning* (ImageNet) y una cabeza nueva de 2 clases con *dropout* (regularización):
* **DenseNet121** (Huang et al., 2017): conexiones densas, ~7 M de parámetros.
* **ResNet50** (He et al., 2016): bloques residuales, ~24 M de parámetros.

Se hace *fine-tuning* de toda la red con AdamW (*weight decay* desacoplado), *cosine annealing* y pérdida de entropía cruzada **ponderada por clase**.

In [ ]:
def build_model(name, pretrained=True):
    if name == "densenet121":
        w = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        m = models.densenet121(weights=w)
        m.classifier = nn.Sequential(nn.Dropout(CFG["DROPOUT"]), nn.Linear(m.classifier.in_features, 2))
    elif name == "resnet50":
        w = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        m = models.resnet50(weights=w)
        m.fc = nn.Sequential(nn.Dropout(CFG["DROPOUT"]), nn.Linear(m.fc.in_features, 2))
    else:
        raise ValueError(name)
    return m

def target_layer(model, name):                          # última capa convolucional para Grad-CAM
    return model.features[-1] if name == "densenet121" else model.layer4[-1]

arch_rows = []
for n in CFG["MODELS"]:
    m = build_model(n, pretrained=False)
    n_par = sum(p.numel() for p in m.parameters())
    head = m.classifier if n == "densenet121" else m.fc
    arch_rows.append(dict(modelo=n, parametros_M=round(n_par / 1e6, 2),
                          capa_Grad_CAM=("features[-1] (norm5)" if n == "densenet121" else "layer4[-1]"),
                          cabeza=str(head).replace("\n", " ")))
display(pd.DataFrame(arch_rows))
PARAMS_M = {r["modelo"]: r["parametros_M"] for r in arch_rows}

## 4. Entrenamiento
Registro de pérdida y exactitud por época, selección del mejor modelo por **AUC de validación** y *early stopping*. El umbral de decisión se elige en validación (índice de Youden) y **se mantiene fijo** al evaluar en el test y en Montgomery.

In [ ]:
def run_epoch(model, loader, criterion, opt=None, scaler=None):
    train = opt is not None
    model.train(train)
    total, n, correct, probs, ys = 0.0, 0, 0, [], []
    if train:
        opt.zero_grad(set_to_none=True)
    with torch.set_grad_enabled(train):
        for step, (x, y) in enumerate(loader):
            x, y = x.to(device), y.to(device)
            with torch.autocast(device_type=device.type, dtype=torch.float16,
                                enabled=train and device.type == "cuda"):
                out = model(x)
                loss = criterion(out, y)
            if train:
                group_start = (step // CFG["ACCUM_STEPS"]) * CFG["ACCUM_STEPS"]
                group_size = min(CFG["ACCUM_STEPS"], len(loader) - group_start)
                scaler.scale(loss / group_size).backward()
                if (step + 1) % CFG["ACCUM_STEPS"] == 0 or step + 1 == len(loader):
                    scaler.step(opt)
                    scaler.update()
                    opt.zero_grad(set_to_none=True)
            total += loss.item() * len(y)
            n += len(y)
            correct += (out.argmax(1) == y).sum().item()
            probs.append(F.softmax(out.float(), 1)[:, 1].detach().cpu().numpy())
            ys.append(y.cpu().numpy())
    probs, ys = np.concatenate(probs), np.concatenate(ys)
    auc = roc_auc_score(ys, probs) if len(np.unique(ys)) > 1 else float("nan")
    return total / n, correct / n, auc

def train_model(name, seed):
    set_seed(seed)
    model = build_model(name, CFG["PRETRAINED"]).to(device)
    cnt = np.bincount(sz_train.label.values, minlength=2)
    weights = torch.tensor(len(sz_train) / (2.0 * cnt), dtype=torch.float32, device=device)
    criterion = nn.CrossEntropyLoss(weight=weights)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["EPOCHS"])
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    dl_tr, dl_va = make_loader(sz_train, True, seed), make_loader(sz_val, False)
    hist = {k: [] for k in ["train_loss", "train_acc", "val_loss", "val_acc", "val_auc"]}
    best_auc, best_ep, bad = -1, 0, 0
    ckpt = OUT / f"{name}_s{seed}.pt"
    for ep in range(CFG["EPOCHS"]):
        tl, ta, _ = run_epoch(model, dl_tr, criterion, opt, scaler)
        sched.step()
        vl, va, vauc = run_epoch(model, dl_va, criterion)
        for k, v in zip(hist, [tl, ta, vl, va, vauc]):
            hist[k].append(float(v))
        if vauc > best_auc:
            best_auc, best_ep, bad = vauc, ep, 0
            torch.save(model.state_dict(), ckpt)
        else:
            bad += 1
        print(f"  [{name} s{seed}] ep {ep+1:02d} | loss {tl:.3f}/{vl:.3f} | acc {ta:.3f}/{va:.3f} | val AUC {vauc:.3f}", flush=True)
        if bad >= CFG["PATIENCE"]:
            print("  early stopping")
            break
    model.load_state_dict(torch.load(ckpt, map_location=device, weights_only=True))
    return model, hist, best_ep + 1

@torch.no_grad()
def predict_proba(model, frame):
    model.eval()
    ps = []
    for x, _ in make_loader(frame, False, bs=8):
        ps.append(F.softmax(model(x.to(device)), 1)[:, 1].cpu().numpy())
    return np.concatenate(ps), frame.label.values

def youden_threshold(y, p):
    fpr, tpr, thr = roc_curve(y, p)
    return float(thr[np.argmax(tpr - fpr)])

def compute_metrics(y, p, thr):
    yhat = (p >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, yhat, labels=[0, 1]).ravel()
    sens, spec = tp / max(tp + fn, 1), tn / max(tn + fp, 1)
    prec = tp / max(tp + fp, 1)
    return dict(acc=(tp + tn) / len(y), sens=sens, spec=spec, prec=prec,
                f1=2 * prec * sens / max(prec + sens, 1e-12), auc=roc_auc_score(y, p))


In [ ]:
RUNS = {}
t_all = time.time()
for mname in CFG["MODELS"]:
    for seed in CFG["SEEDS"]:
        t0 = time.time(); print(f"\n=== {mname} | semilla {seed} ===")
        model, hist, best_ep = train_model(mname, seed)
        pv, yv = predict_proba(model, sz_val); thr = youden_threshold(yv, pv)
        pt, yt = predict_proba(model, sz_test); pm, ym = predict_proba(model, mg)
        RUNS[(mname, seed)] = dict(hist=hist, thr=thr, best_epoch=best_ep,
                                   p_sz=pt, y_sz=yt, p_mg=pm, y_mg=ym,
                                   m_sz=compute_metrics(yt, pt, thr), m_mg=compute_metrics(ym, pm, thr))
        r = RUNS[(mname, seed)]
        print(f"  -> thr={thr:.3f} | Shenzhen AUC {r['m_sz']['auc']:.3f} F1 {r['m_sz']['f1']:.3f} | "
              f"Montgomery AUC {r['m_mg']['auc']:.3f} F1 {r['m_mg']['f1']:.3f} | {time.time()-t0:.0f}s")
        del model; torch.cuda.empty_cache()
print(f"\nTiempo total de entrenamiento: {(time.time()-t_all)/60:.1f} min")

## 5. Evaluación
Métricas en el test interno (Shenzhen) y en la validación externa (Montgomery), media ± desviación estándar sobre las semillas. **Sens.** = sensibilidad (recall de TB), **Espec.** = especificidad.

In [ ]:
METRICS = [("acc", "Acc"), ("f1", "F1"), ("sens", "Sens."), ("spec", "Espec."), ("auc", "AUC")]
MODEL_LABEL = {"densenet121": "DenseNet121", "resnet50": "ResNet50"}
SET_LABEL = {"sz": "Shenzhen (test)", "mg": "Montgomery (ext.)"}

def perf_stats(mname, sset):
    vals = {k: np.array([RUNS[(mname, s)][f"m_{sset}"][k] for s in CFG["SEEDS"]]) for k, _ in METRICS}
    return {k: (v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0) for k, v in vals.items()}

perf_rows = []
for mname in CFG["MODELS"]:
    for sset in ("sz", "mg"):
        st = perf_stats(mname, sset)
        perf_rows.append({"Modelo": MODEL_LABEL[mname], "Conjunto": SET_LABEL[sset],
                          **{lab: f"{st[k][0]:.3f} ± {st[k][1]:.3f}" for k, lab in METRICS}})
perf_df = pd.DataFrame(perf_rows); display(perf_df)

In [ ]:
SEED0 = CFG["SEEDS"][0]
colors = {"densenet121": "tab:blue", "resnet50": "tab:red"}

# (a) Curvas de entrenamiento (semilla 0)
fig, axs = plt.subplots(1, 2, figsize=(9, 3.2))
for mname in CFG["MODELS"]:
    h = RUNS[(mname, SEED0)]["hist"]; ep = np.arange(1, len(h["train_loss"]) + 1)
    axs[0].plot(ep, h["train_loss"], "--", c=colors[mname], label=f"{MODEL_LABEL[mname]} train")
    axs[0].plot(ep, h["val_loss"], "-", c=colors[mname], label=f"{MODEL_LABEL[mname]} val")
    axs[1].plot(ep, h["val_auc"], "-o", ms=3, c=colors[mname], label=MODEL_LABEL[mname])
axs[0].set(xlabel="Época", ylabel="Pérdida (CE ponderada)", title="Pérdida"); axs[0].legend(fontsize=7)
axs[1].set(xlabel="Época", ylabel="AUC de validación", title="AUC en validación"); axs[1].legend(fontsize=7)
for a in axs: a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(OUT / "fig_curves.png", dpi=200); plt.show()

# (b) Curvas ROC (semilla 0): interno vs. externo
fig, axs = plt.subplots(1, 2, figsize=(8, 3.4), sharey=True)
for ax, sset in zip(axs, ("sz", "mg")):
    for mname in CFG["MODELS"]:
        r = RUNS[(mname, SEED0)]; fpr, tpr, _ = roc_curve(r[f"y_{sset}"], r[f"p_{sset}"])
        ax.plot(fpr, tpr, c=colors[mname], label=f"{MODEL_LABEL[mname]} (AUC={r[f'm_{sset}']['auc']:.3f})")
    ax.plot([0, 1], [0, 1], "k:", lw=.8); ax.set(title=SET_LABEL[sset], xlabel="1 − Especificidad"); ax.grid(alpha=.3); ax.legend(fontsize=7, loc="lower right")
axs[0].set_ylabel("Sensibilidad")
plt.tight_layout(); plt.savefig(OUT / "fig_roc.png", dpi=200); plt.show()

# (c) Matrices de confusión (semilla 0)
fig, axs = plt.subplots(2, 2, figsize=(6, 5.2))
for i, mname in enumerate(CFG["MODELS"]):
    for j, sset in enumerate(("sz", "mg")):
        r = RUNS[(mname, SEED0)]
        ConfusionMatrixDisplay(confusion_matrix(r[f"y_{sset}"], (r[f"p_{sset}"] >= r["thr"]).astype(int), labels=[0, 1]),
                               display_labels=["Normal", "TB"]).plot(ax=axs[i, j], colorbar=False, cmap="Blues")
        axs[i, j].set_title(f"{MODEL_LABEL[mname]} · {SET_LABEL[sset]}", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "fig_confusion.png", dpi=200); plt.show()

## 6. Explicabilidad (XAI): ¿mira el modelo los pulmones?
Se explican los modelos de la semilla 0, siempre para la **clase predicha**:
* **Grad-CAM** (Selvaraju et al., 2017): gradientes de la clase respecto a la última capa convolucional (`pytorch-grad-cam`).
* **LIME** (Ribeiro et al., 2016): sustituto lineal local sobre superpíxeles SLIC; el mapa conserva los pesos positivos (evidencia a favor de la clase predicha).
* **Aleatorio**: mapa de ruido suavizado (hipótesis nula).

**Métricas cuantitativas** (mapa normalizado a [0, 1]):
| Métrica | Definición | Lectura |
|---|---|---|
| **EIL** (*energy in lung*) | fracción de la energía del mapa dentro de la máscara de pulmón | más alto = más centrado en el pulmón |
| **Ganancia** | EIL / fracción de área pulmonar | 1 = como un mapa uniforme; > 1 = prefiere el pulmón |
| **Puntería** | ¿el máximo del mapa cae dentro del pulmón? | proporción de aciertos |
| **Borde** | energía en el marco exterior (5 %) / área del marco | > 1 = el modelo atiende a los márgenes (posibles artefactos); no necesita máscara |
| **Deletion AUC** (fidelidad) | probabilidad media de la clase al ocultar el 0–50 % de píxeles más relevantes según el mapa | más bajo = explicación más fiel (Petsiuk et al., 2018) |

Las métricas con máscara solo se pueden calcular en Montgomery; **Borde** también en el test de Shenzhen.

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from lime import lime_image
from lime.wrappers.scikit_image import SegmentationAlgorithm

MEAN_T = torch.tensor(IMAGENET_MEAN, device=device).view(1, 3, 1, 1)
STD_T = torch.tensor(IMAGENET_STD, device=device).view(1, 3, 1, 1)
BW = int(0.05 * S); FRAME = np.ones((S, S), bool); FRAME[BW:-BW, BW:-BW] = False; FRAME_FRAC = FRAME.mean()

def load_trained(mname, seed):
    m = build_model(mname, pretrained=False)
    m.load_state_dict(torch.load(OUT / f"{mname}_s{seed}.pt", map_location=device, weights_only=True))
    return m.to(device).eval()

@torch.no_grad()
def class_probs(model, X):
    return torch.cat([F.softmax(model(X[i:i + 8].to(device)), 1).cpu() for i in range(0, len(X), 8)]).numpy()

def gradcam_maps(model, mname, X, classes, bs=2):
    cam = GradCAM(model=model, target_layers=[target_layer(model, mname)])
    out = []
    for i in range(0, len(X), bs):
        tg = [ClassifierOutputTarget(int(c)) for c in classes[i:i + bs]]
        out.append(cam(input_tensor=X[i:i + bs].to(device), targets=tg))
    try: cam.activations_and_grads.release()
    except Exception: pass
    return np.concatenate(out)                           # (N, S, S) en [0, 1]

def lime_maps(model, frame, classes, n_samples):
    @torch.no_grad()
    def predict_fn(batch):                               # batch: (n, H, W, 3) uint8 -> probabilidades
        x = torch.from_numpy(np.asarray(batch, dtype=np.float32) / 255.0).permute(0, 3, 1, 2).to(device)
        return F.softmax(model((x - MEAN_T) / STD_T), 1).cpu().numpy()
    explainer = lime_image.LimeImageExplainer(random_state=0)
    seg = SegmentationAlgorithm("slic", n_segments=60, compactness=10, sigma=1, start_label=1)
    maps = []
    for path, c in tqdm(list(zip(frame.path, classes)), desc="LIME", leave=False):
        rgb = np.stack([np.array(IMG_CACHE[path])] * 3, -1).astype(np.uint8)
        kw = dict(top_labels=2, hide_color=0, num_samples=n_samples, batch_size=8, segmentation_fn=seg)
        try:
            exp = explainer.explain_instance(rgb, predict_fn, progress_bar=False, **kw)
        except TypeError:                                # versiones antiguas de lime no tienen progress_bar
            exp = explainer.explain_instance(rgb, predict_fn, **kw)
        w = dict(exp.local_exp[int(c)]); h = np.zeros(exp.segments.shape, np.float32)
        for sid in np.unique(exp.segments): h[exp.segments == sid] = max(w.get(sid, 0.0), 0.0)
        maps.append(h / h.max() if h.max() > 0 else h)
    return np.stack(maps)

def random_maps(n, seed=0):
    rng = np.random.RandomState(seed); out = []
    for _ in range(n):
        h = cv2.resize(rng.rand(7, 7).astype(np.float32), (S, S), interpolation=cv2.INTER_LINEAR)
        out.append((h - h.min()) / (h.max() - h.min() + 1e-12))
    return np.stack(out)

def score_map(h, mask=None):
    h = np.clip(h, 0, None); tot = h.sum()
    area = float(mask.mean()) if mask is not None else np.nan
    if tot <= 1e-12:
        return dict(border=np.nan, eil=np.nan, area=area, gain=np.nan, point=np.nan, valid=False)
    r = dict(border=(h[FRAME].sum() / tot) / FRAME_FRAC, eil=np.nan, area=area, gain=np.nan, point=np.nan, valid=True)
    if mask is not None:
        if area <= 0:
            raise ValueError("M?scara pulmonar vac?a")
        eil = h[mask].sum() / tot
        r.update(eil=eil, gain=eil / area, point=float(mask[np.unravel_index(h.argmax(), h.shape)]))
    return r

@torch.no_grad()
def deletion_auc(model, x, h, cls, fracs=np.linspace(0, 0.5, 11), seed=0):
    if np.max(h) <= 0:
        return float("nan")
    rng = np.random.RandomState(seed)
    gray = (x[0].cpu().numpy() * IMAGENET_STD[0] + IMAGENET_MEAN[0]).astype(np.float32)
    blurred = cv2.GaussianBlur(gray, (0, 0), sigmaX=10)
    background = torch.stack([(torch.from_numpy(blurred) - IMAGENET_MEAN[c]) / IMAGENET_STD[c] for c in range(3)])
    order = np.argsort(-(h.ravel() + 1e-6 * rng.rand(h.size)))        # más relevante primero (desempate aleatorio)
    batch = []
    for f in fracs:
        xx = x.clone(); k = int(f * h.size)
        if k > 0:
            rr, cc = np.unravel_index(order[:k], h.shape); xx[:, rr, cc] = background[:, rr, cc]   # fondo gris desenfocado, normalizado por canal
        batch.append(xx)
    p = F.softmax(model(torch.stack(batch).to(device)), 1)[:, cls].cpu().numpy()
    return float(np.sum((p[1:] + p[:-1]) / 2 * np.diff(fracs)) / (fracs[-1] - fracs[0]))

In [ ]:
# Subconjunto de Montgomery (equilibrado por clase) para LIME + fidelidad
rng = np.random.RandomState(0); ids = []
for c in (0, 1):
    pool = mg.index[mg.label == c].values
    ids += rng.choice(pool, size=min(CFG["N_XAI"] // 2, len(pool)), replace=False).tolist()
mg_sub = mg.loc[ids].reset_index(drop=True)
print("Subconjunto XAI:", len(mg_sub), "imágenes |", mg_sub.label.value_counts().to_dict())

X_mg, X_sub, X_szt = get_tensor(mg), get_tensor(mg_sub), get_tensor(sz_test)
XAI = {}           # resultados por modelo
sub_rows, full_rows = [], []

for mname in CFG["MODELS"]:
    t0 = time.time(); model = load_trained(mname, SEED0); thr = RUNS[(mname, SEED0)]["thr"]
    pred = lambda X: (class_probs(model, X)[:, 1] >= thr).astype(int)       # clase predicha con el umbral de validación
    c_sub, c_mg, c_szt = pred(X_sub), pred(X_mg), pred(X_szt)

    # --- Subconjunto: Grad-CAM, LIME y aleatorio + fidelidad (deletion) ---
    maps = {"Grad-CAM": gradcam_maps(model, mname, X_sub, c_sub),
            "LIME": lime_maps(model, mg_sub, c_sub, CFG["LIME_SAMPLES"]),
            "Aleatorio": random_maps(len(mg_sub))}
    for meth, M in maps.items():
        for i in range(len(mg_sub)):
            sc = score_map(M[i], MASK_CACHE.get(mg_sub.name[i]))
            sc.update(model=mname, method=meth, name=mg_sub.name[i], label=int(mg_sub.label[i]), pred=int(c_sub[i]),
                      deletion=deletion_auc(model, X_sub[i], M[i], int(c_sub[i])))
            sub_rows.append(sc)

    # --- Grad-CAM sobre TODO Montgomery y el test de Shenzhen ---
    G_mg = gradcam_maps(model, mname, X_mg, c_mg); G_sz = gradcam_maps(model, mname, X_szt, c_szt)
    for i in range(len(mg)):
        sc = score_map(G_mg[i], MASK_CACHE.get(mg.name[i])); sc.update(model=mname, set="mg", name=mg.name[i], label=int(mg.label[i]), pred=int(c_mg[i])); full_rows.append(sc)
    for i in range(len(sz_test)):
        sc = score_map(G_sz[i], None); sc.update(model=mname, set="sz", name=sz_test.name[i], label=int(sz_test.label[i]), pred=int(c_szt[i])); full_rows.append(sc)

    XAI[mname] = dict(maps=maps, G_mg=G_mg, c_sub=c_sub)
    print(f"{mname}: XAI listo en {time.time()-t0:.0f}s"); del model; torch.cuda.empty_cache()

sub_df, full_df = pd.DataFrame(sub_rows), pd.DataFrame(full_rows)
full_df["correct"] = full_df.pred == full_df.label; sub_df["correct"] = sub_df.pred == sub_df.label

In [ ]:
def mean_sd(s):
    s = pd.Series(s).dropna(); return (s.mean(), s.std(ddof=1) if len(s) > 1 else 0.0) if len(s) else (np.nan, np.nan)
fmt = lambda t, d=2: "--" if np.isnan(t[0]) else f"{t[0]:.{d}f} ± {t[1]:.{d}f}"

XAI_TABLE = []                                                       # filas para pantalla y LaTeX
for mname in CFG["MODELS"]:
    for meth in ["Grad-CAM", "LIME", "Aleatorio"]:
        d = sub_df[(sub_df.model == mname) & (sub_df.method == meth)]
        XAI_TABLE.append(dict(Modelo=MODEL_LABEL[mname], Método=meth, n=int(d.border.notna().sum()), EIL=mean_sd(d.eil), Ganancia=mean_sd(d.gain),
                              Puntería=(d.point.mean(), 0.0), Borde=mean_sd(d.border), Deletion=mean_sd(d.deletion)))
    for sset, lab in (("mg", "Grad-CAM (Mont. completo)"), ("sz", "Grad-CAM (Shen. test)")):
        d = full_df[(full_df.model == mname) & (full_df.set == sset)]
        XAI_TABLE.append(dict(Modelo=MODEL_LABEL[mname], Método=lab, n=int(d.border.notna().sum()), EIL=mean_sd(d.eil), Ganancia=mean_sd(d.gain),
                              Puntería=(d.point.mean() if sset == "mg" else np.nan, 0.0), Borde=mean_sd(d.border), Deletion=(np.nan, np.nan)))

disp = pd.DataFrame([{**{k: v for k, v in r.items() if k in ("Modelo", "Método", "n")},
                      "EIL": fmt(r["EIL"], 3), "Ganancia": fmt(r["Ganancia"]),
                      "Puntería": "--" if np.isnan(r["Puntería"][0]) else f"{r['Puntería'][0]:.2f}",
                      "Borde": fmt(r["Borde"]), "Deletion↓": fmt(r["Deletion"], 3)} for r in XAI_TABLE])
display(disp)
area_mean = float(np.mean([m.mean() for m in MASK_CACHE.values()]))
print(f"Fracción media de área pulmonar en Montgomery (nivel del azar para EIL): {area_mean:.3f}")

# Pruebas estadísticas
STATS = {}
for mname in CFG["MODELS"]:
    g = sub_df[(sub_df.model == mname) & (sub_df.method == "Grad-CAM")].set_index("name").eil
    l = sub_df[(sub_df.model == mname) & (sub_df.method == "LIME")].set_index("name").eil.reindex(g.index)
    pairs = pd.concat([g.rename("gradcam"), l.rename("lime")], axis=1).dropna()
    p_w = stats.wilcoxon(pairs.gradcam, pairs.lime).pvalue if len(pairs) > 5 and (pairs.gradcam - pairs.lime).abs().sum() > 0 else float("nan")
    f = full_df[(full_df.model == mname) & (full_df.set == "mg")]
    ok, err = f[f.correct].eil.dropna(), f[~f.correct].eil.dropna()
    p_mw = stats.mannwhitneyu(ok, err).pvalue if len(err) >= 3 else float("nan")
    STATS[mname] = dict(p_wilcoxon=p_w, p_mw=p_mw, eil_ok=ok.mean(), eil_err=err.mean() if len(err) else np.nan, n_ok=len(ok), n_err=len(err))
    print(f"{MODEL_LABEL[mname]}: Wilcoxon Grad-CAM vs LIME (EIL) p={p_w:.4f} | aciertos n={len(ok)} EIL={ok.mean():.3f} | "
          f"errores n={len(err)} EIL={(err.mean() if len(err) else float('nan')):.3f} | Mann-Whitney p={p_mw:.4f}")

# EIL por clase real (Grad-CAM, Montgomery completo)
display(full_df[full_df.set == "mg"].groupby(["model", "label"]).agg(n=("eil", "size"), EIL=("eil", "mean"),
        Ganancia=("gain", "mean"), Borde=("border", "mean")).rename(index={0: "Normal", 1: "TB"}, level=1))

In [ ]:
# --- Figura cualitativa: Grad-CAM y LIME (modelo 1) ---
def overlay(ax, img, h, mask=None, title=""):
    ax.imshow(img, cmap="gray"); ax.imshow(h, cmap="jet", alpha=np.clip(h, 0, 1) * 0.55, vmin=0, vmax=1)
    if mask is not None: ax.contour(mask, levels=[0.5], colors="white", linewidths=0.7)
    ax.set_title(title, fontsize=7); ax.axis("off")

mname = CFG["MODELS"][0]; maps = XAI[mname]["maps"]; c_sub = XAI[mname]["c_sub"]
tb_ok = np.where((mg_sub.label == 1) & (c_sub == 1))[0]; nm_ok = np.where((mg_sub.label == 0) & (c_sub == 0))[0]
wrong = np.where(mg_sub.label.values != c_sub)[0]
picks = [("TB acertado", tb_ok[0] if len(tb_ok) else None), ("Normal acertado", nm_ok[0] if len(nm_ok) else None),
         ("Error", wrong[0] if len(wrong) else (tb_ok[1] if len(tb_ok) > 1 else None))]
picks = [(t, i) for t, i in picks if i is not None]
fig, axs = plt.subplots(len(picks), 3, figsize=(5.2, 1.8 * len(picks)))
axs = np.atleast_2d(axs)
for r, (tag, i) in enumerate(picks):
    nm = mg_sub.name[i]; mk = MASK_CACHE.get(nm); img = IMG_CACHE[mg_sub.path[i]]
    lab = f"real={'TB' if mg_sub.label[i] else 'N'}, pred={'TB' if c_sub[i] else 'N'}"
    axs[r, 0].imshow(img, cmap="gray"); axs[r, 0].contour(mk, levels=[0.5], colors="yellow", linewidths=0.7) if mk is not None else None
    axs[r, 0].set_title(f"{tag}\n{lab}", fontsize=7); axs[r, 0].axis("off")
    overlay(axs[r, 1], img, maps["Grad-CAM"][i], mk, "Grad-CAM"); overlay(axs[r, 2], img, maps["LIME"][i], mk, "LIME")
plt.tight_layout(); plt.savefig(OUT / "fig_cam_examples.png", dpi=220); plt.show()

# --- Figura cuantitativa: EIL por método (el punteado es el nivel del azar) ---
fig, axs = plt.subplots(1, 2, figsize=(5.4, 2.6), sharey=True)
for ax, mname in zip(axs, CFG["MODELS"]):
    data = [sub_df[(sub_df.model == mname) & (sub_df.method == m)].eil.dropna().values for m in ["Grad-CAM", "LIME", "Aleatorio"]]
    ax.boxplot(data, showmeans=True); ax.set_xticklabels(["Grad-CAM", "LIME", "Aleat."])
    ax.axhline(area_mean, ls="--", c="gray", lw=1); ax.set_title(MODEL_LABEL[mname], fontsize=8); ax.tick_params(labelsize=7); ax.grid(alpha=.3)
axs[0].set_ylabel("EIL (energía en pulmón)", fontsize=8)
plt.tight_layout(); plt.savefig(OUT / "fig_xai_box.png", dpi=220); plt.show()

## 7. Localizaci?n de regiones compatibles con TB

An?lisis exploratorio sobre las 55 radiograf?as positivas de Montgomery con cajas de consenso NLM. Se calcula Grad-CAM para la clase TB aun cuando la predicci?n sea negativa; se compara la energ?a dentro de las cajas con un mapa aleatorio, condicionando por el ?rea pulmonar. Las semillas se agregan por imagen antes de estimar intervalos de confianza.


In [ ]:
from scripts.lesion_roi import run_lesion_analysis
LESION_ROI = run_lesion_analysis(csv_path="data/montgomery_consensus_roi.csv", mg=mg, mask_cache=MASK_CACHE,
    get_tensor=get_tensor, load_trained=load_trained, gradcam_maps=gradcam_maps,
    random_maps=random_maps, class_probs=class_probs, model_names=CFG["MODELS"],
    seeds=CFG["SEEDS"], size=S, out_dir=OUT)
display(pd.DataFrame(LESION_ROI["models"]).T)


## 7. Exportación para el artículo
Se generan las tablas LaTeX, un archivo de macros con **todas las cifras** del artículo y un `.zip` con los resultados. Copia la carpeta `results/` junto a `articulo_ieee.tex` y compila: los números del artículo se actualizan solos.

In [ ]:
MK = {"densenet121": "Dense", "resnet50": "Res"}; SK = {"sz": "Sz", "mg": "Mg"}
MACROS = {}
pm = lambda m, s, d=2: f"{m:.{d}f}$\\pm${s:.{d}f}"

# Datos
MACROS.update(nSzTrain=len(sz_train), nSzVal=len(sz_val), nSzTest=len(sz_test), nSzAll=len(sz), nMg=len(mg),
              nSzTB=int(sz.label.sum()), nMgTB=int(mg.label.sum()), nMask=len(MASK_CACHE),
              nSub=len(mg_sub), nSeeds=len(CFG["SEEDS"]), nEpochs=CFG["EPOCHS"], areaLung=f"{area_mean:.2f}")
# Rendimiento
for mname in CFG["MODELS"]:
    MACROS[f"params{MK[mname]}"] = f"{PARAMS_M[mname]:.1f}"
    MACROS[f"thr{MK[mname]}"] = f"{RUNS[(mname, SEED0)]['thr']:.2f}"
    for sset in ("sz", "mg"):
        st = perf_stats(mname, sset)
        for k, nm in [("acc", "Acc"), ("f1", "Fs"), ("sens", "Sens"), ("spec", "Spec"), ("auc", "Auc")]:
            MACROS[f"perf{MK[mname]}{SK[sset]}{nm}"] = pm(*st[k], d=3 if k == "auc" else 2)
# XAI
for r in XAI_TABLE:
    mk = "Dense" if r["Modelo"] == "DenseNet121" else "Res"
    key = {"Grad-CAM": "Gc", "LIME": "Lime", "Aleatorio": "Rnd", "Grad-CAM (Mont. completo)": "GcMg", "Grad-CAM (Shen. test)": "GcSz"}[r["Método"]]
    for fld, nm, d in [("EIL", "Eil", 3), ("Ganancia", "Gain", 2), ("Borde", "Bord", 2), ("Deletion", "Del", 3)]:
        if not np.isnan(r[fld][0]): MACROS[f"xai{mk}{key}{nm}"] = pm(*r[fld], d=d)
    if not np.isnan(r["Puntería"][0]): MACROS[f"xai{mk}{key}Pt"] = f"{r['Puntería'][0]:.2f}"
for mname in CFG["MODELS"]:
    s = STATS[mname]; mk = MK[mname]
    MACROS[f"eilOk{mk}"] = f"{s['eil_ok']:.3f}"; MACROS[f"eilErr{mk}"] = "--" if np.isnan(s["eil_err"]) else f"{s['eil_err']:.3f}"
    MACROS[f"nOk{mk}"] = s["n_ok"]; MACROS[f"nErr{mk}"] = s["n_err"]
    MACROS[f"pWilc{mk}"] = "--" if np.isnan(s["p_wilcoxon"]) else (f"{s['p_wilcoxon']:.3f}" if s["p_wilcoxon"] >= 0.001 else "<0.001")
    MACROS[f"pMw{mk}"] = "--" if np.isnan(s["p_mw"]) else f"{s['p_mw']:.3f}"

with open(OUT / "macros.tex", "w", encoding="utf-8") as f:
    f.write("% Generado automáticamente por el notebook\n")
    for k, v in MACROS.items(): f.write(f"\\def\\{k}{{{v}}}\n")

# Tabla de rendimiento (LaTeX)
L = ["\\begin{tabular}{llccccc}", "\\toprule", "Modelo & Conjunto & Acc & F1 & Sens & Espec & AUC \\\\", "\\midrule"]
for mname in CFG["MODELS"]:
    for sset in ("sz", "mg"):
        st = perf_stats(mname, sset)
        L.append(f"{MODEL_LABEL[mname]} & {SET_LABEL[sset]} & " + " & ".join(pm(*st[k], d=3 if k == "auc" else 2) for k in ["acc", "f1", "sens", "spec", "auc"]) + " \\\\")
    if mname != CFG["MODELS"][-1]: L.append("\\midrule")
L += ["\\bottomrule", "\\end{tabular}"]
(OUT / "tab_perf.tex").write_text("\n".join(L), encoding="utf-8")

# Tabla XAI (LaTeX)
cell = lambda t, d=2: "--" if np.isnan(t[0]) else pm(*t, d=d)
L = ["\\begin{tabular}{llcccccc}", "\\toprule", "Modelo & Método & $n$ & EIL & Ganancia & Punt. & Borde & Del.$\\downarrow$ \\\\", "\\midrule"]
for i, r in enumerate(XAI_TABLE):
    pt = "--" if np.isnan(r["Puntería"][0]) else f"{r['Puntería'][0]:.2f}"
    L.append(f"{r['Modelo']} & {r['Método']} & {r['n']} & {cell(r['EIL'], 3)} & {cell(r['Ganancia'])} & {pt} & {cell(r['Borde'])} & {cell(r['Deletion'], 3)} \\\\")
    if (i + 1) % 5 == 0 and i + 1 < len(XAI_TABLE): L.append("\\midrule")
L += ["\\bottomrule", "\\end{tabular}"]
(OUT / "tab_xai.tex").write_text("\n".join(L), encoding="utf-8")

# Registro de configuración y versiones + zip
json.dump(dict(cfg={k: v for k, v in CFG.items()}, torch=torch.__version__, torchvision=torchvision.__version__,
               python=platform.python_version(), device=str(device),
               gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
               thresholds={f"{m}_s{s}": RUNS[(m, s)]["thr"] for (m, s) in RUNS},
               best_epochs={f"{m}_s{s}": RUNS[(m, s)]["best_epoch"] for (m, s) in RUNS}),
          open(OUT / "config.json", "w", encoding="utf-8"), indent=1)
sub_df.to_csv(OUT / "xai_subset_scores.csv", index=False); full_df.to_csv(OUT / "xai_full_scores.csv", index=False)
perf_df.to_csv(OUT / "performance.csv", index=False)
PREDICTIONS = []
for (mname, seed), run in RUNS.items():
    for sset, frame in (("sz", sz_test), ("mg", mg)):
        for i, item in frame.iterrows():
            prob = float(run[f"p_{sset}"][i])
            PREDICTIONS.append(dict(model=mname, seed=seed, dataset=sset, name=item["name"],
                                    label=int(item["label"]), probability_tb=prob,
                                    threshold=run["thr"], predicted=int(prob >= run["thr"])))
pd.DataFrame(PREDICTIONS).to_csv(OUT / "predictions.csv", index=False)
(OUT / "training_history.json").write_text(json.dumps({f"{m}_s{s}": RUNS[(m, s)]["hist"] for m, s in RUNS}, indent=2), encoding="utf-8")

zip_path = OUT.parent / f"{OUT.name}.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.glob("*")):
        if p.suffix != ".pt": z.write(p, p.name)               # los pesos (.pt) se omiten del zip por tamaño
print("Archivos en results/:", sorted(p.name for p in OUT.glob("*") if p.suffix != ".pt"))
print("Macros exportadas:", len(MACROS))
try:
    from google.colab import files; files.download(str(zip_path))  # descarga automática en Colab
except Exception:
    pass

### Notas y limitaciones (para la discusión del artículo)
* **Tamaño de muestra pequeño** (≈660 + 138 imágenes): las métricas tienen incertidumbre; por eso se reportan 3 semillas y desviaciones estándar.
* Las máscaras de pulmón existen solo en Montgomery, así que EIL/Ganancia/Puntería se miden allí; en Shenzhen solo la métrica de **Borde**.
* Grad-CAM tiene baja resolución espacial (7×7 interpolado) y LIME depende de la segmentación en superpíxeles y del muestreo (varianza): ninguna explicación es una verdad absoluta (ver *sanity checks*, Adebayo et al., 2018).
* Que el mapa caiga sobre el pulmón **no demuestra** que el razonamiento sea clínicamente correcto (el modelo puede usar señales sutiles del contexto dentro del pulmón).
* **Ejercicio académico:** no es un dispositivo clínico ni validado para diagnóstico.